In [1]:
import sys

In [2]:
# !{sys.executable} -m pip install jupysql

In [3]:
import sqlite3

Schema

tabla1 company_media_spend

tabla2 company_transactions

tabla3 users

```bash
sqlite3 test.db
.mode csv
.import company_media_spend.csv tabla1
.import company_transactions.csv tabla2
.import users.csv tabla3
.quite
```

In [4]:
%load_ext sql
%sql sqlite:///test.db
%config SqlMagic.displaylimit = None

Connecting to 'sqlite:///test.db'

displaylimit: Value None will be treated as 0 (no limit)

## 1. Transacciones exitosas y volumen por mes

In [5]:
%%sql
SELECT
    strftime('%Y-%m', created_at) AS month,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_usd), 2) AS total_volume_usd
FROM tabla2
WHERE status IN ('COMPLETE', 'PAID', 'PAYABLE')
GROUP BY month
ORDER BY month;

Running query in 'sqlite:///test.db'

month,total_transactions,total_volume_usd
2024-01,281,15976.25
2024-02,366,20752.23
2024-03,438,25901.2
2024-04,439,25800.98
2024-05,498,29679.16
2024-06,527,31398.15
2024-07,507,30446.58
2024-08,531,31626.22
2024-09,531,32288.61
2024-10,539,31624.45


## 2. Conversión por canal de adquisición

In [6]:
%%sql
SELECT
    u.acquisition_channel,
    COUNT(DISTINCT u.user_id) AS users_acquired,
    COUNT(DISTINCT t.user_id) AS users_converted,
    ROUND(COUNT(DISTINCT t.user_id) * 100.0 / NULLIF(COUNT(DISTINCT u.user_id), 0),2) AS conversion_rate_pct
FROM tabla3 u
LEFT JOIN tabla2 t
    ON u.user_id = t.user_id
    AND t.status IN ('COMPLETE', 'PAID', 'PAYABLE')
GROUP BY u.acquisition_channel
ORDER BY users_acquired DESC;

Running query in 'sqlite:///test.db'

acquisition_channel,users_acquired,users_converted,conversion_rate_pct
paid_meta,1010,1010,100.0
paid_google,570,570,100.0
organic_search,538,538,100.0
referral,384,384,100.0
direct,324,324,100.0
email,174,174,100.0


## 3. Recurrentes por mes

In [7]:
%%sql
WITH first_transaction AS (
    SELECT
        user_id,
        strftime('%Y-%m', MIN(created_at)) AS first_month
    FROM tabla2
    WHERE status IN ('COMPLETE', 'PAID', 'PAYABLE')
    GROUP BY user_id
),
monthly_active AS (
    SELECT DISTINCT
        user_id,
        strftime('%Y-%m', created_at) AS active_month
    FROM tabla2
    WHERE status IN ('COMPLETE', 'PAID', 'PAYABLE')
)
SELECT
    ma.active_month,
    COUNT(DISTINCT CASE WHEN ma.active_month =  ft.first_month THEN ma.user_id END) AS new_users,
    COUNT(DISTINCT CASE WHEN ma.active_month != ft.first_month THEN ma.user_id END) AS returning_users,
    COUNT(DISTINCT ma.user_id) AS total_mau
FROM monthly_active ma
JOIN first_transaction ft ON ma.user_id = ft.user_id
GROUP BY ma.active_month
ORDER BY ma.active_month;

Running query in 'sqlite:///test.db'

active_month,new_users,returning_users,total_mau
2024-01,267,0,267
2024-02,282,67,349
2024-03,269,139,408
2024-04,269,151,420
2024-05,278,191,469
2024-06,293,198,491
2024-07,261,213,474
2024-08,275,218,493
2024-09,262,225,487
2024-10,273,231,504


## 4. Por canal pagado

In [8]:
%%sql
WITH spend AS (
    SELECT
        channel,
        SUM(spend_usd) AS total_spend_usd
    FROM tabla1
    GROUP BY channel
),
revenue AS (
    SELECT
        u.acquisition_channel AS channel,
        SUM(t.amount_usd) AS total_revenue_usd
    FROM tabla3 u
    JOIN tabla2 t
        ON u.user_id = t.user_id
        AND t.status IN ('COMPLETE', 'PAID', 'PAYABLE')
    GROUP BY u.acquisition_channel
)
SELECT
    s.channel,
    ROUND(s.total_spend_usd, 2) AS total_spend_usd,
    ROUND(r.total_revenue_usd, 2) AS total_revenue_usd,
    ROUND(r.total_revenue_usd / NULLIF(s.total_spend_usd, 0), 2) AS roas
FROM spend s
LEFT JOIN revenue r ON s.channel = r.channel
ORDER BY roas DESC NULLS LAST;

Running query in 'sqlite:///test.db'

channel,total_spend_usd,total_revenue_usd,roas
email,77872.47,18706.86,0.24
paid_google,646036.08,60020.15,0.09
paid_meta,1071615.01,55659.4,0.05


## 5. Retención de 30 días por canal

In [9]:
%%sql
WITH user_first_tx AS (
    SELECT
        t.user_id,
        u.acquisition_channel,
        MIN(t.created_at) AS first_tx_date
    FROM tabla3 u
    JOIN tabla2 t
        ON u.user_id = t.user_id
        AND t.status IN ('COMPLETE', 'PAID', 'PAYABLE')
    GROUP BY t.user_id, u.acquisition_channel
),
second_tx AS (
    SELECT DISTINCT
        f.user_id,
        f.acquisition_channel
    FROM user_first_tx f
    JOIN tabla2 t
        ON f.user_id = t.user_id
        AND t.status IN ('COMPLETE', 'PAID', 'PAYABLE')
        AND t.created_at > f.first_tx_date
        AND julianday(t.created_at) - julianday(f.first_tx_date) <= 30
)
SELECT
    f.acquisition_channel,
    COUNT(DISTINCT f.user_id) AS users_with_first_tx,
    COUNT(DISTINCT s.user_id) AS retained_30d,
    ROUND(COUNT(DISTINCT s.user_id) * 100.0 / NULLIF(COUNT(DISTINCT f.user_id), 0), 2) AS retention_30d_pct
FROM user_first_tx f
LEFT JOIN second_tx s ON f.user_id = s.user_id
GROUP BY f.acquisition_channel
ORDER BY retention_30d_pct DESC NULLS LAST;

Running query in 'sqlite:///test.db'

acquisition_channel,users_with_first_tx,retained_30d,retention_30d_pct
organic_search,538,162,30.11
referral,384,104,27.08
direct,324,74,22.84
email,174,38,21.84
paid_google,570,113,19.82
paid_meta,1010,76,7.52
